In [1]:
import matplotlib.pyplot as plt
import numpy as np

##### Как пользоваться
Эта программака подгружает координаты (двумерные x-y) аэродинамического профиля. Ниже в ячейке надо указать путь до txt файла с координатами
Сами файлы с координатами можно скачать на сайте [AirfoilTools](http://airfoiltools.com/)

In [3]:
Coord = np.loadtxt('C:/Users/PC/Desktop/Shchegly_SKAT26/Модель/КоординатыПрофилей/SD8020.txt', dtype=float)

##### Структура файла с координатами
В файле с координатами при копировании с сайта [AirfoilTools](http://airfoiltools.com/) в первой строчке указано название профиля
По умолчанию надо убрать это название, оставив только координаты, Solid Works тоже в конечно счете примет файд, содержащий только координаты
профиля.

Ориентация профиля и системы координат относительного него следующая:
![.](AirfoilOrientation.png)

##### Функции
Тут есть три функции: масштабирование, смещение и поворот:

Scale(cord, chord) принимает подгруженный ранее массив координат профиля двумерного **cord**, второй параметр **chord** - желаемая длина хорды отмасштабированного профиля в миллиметрах. На выходе функции  - массив координат отмасштабированного профиля с добавленной компонетной по третьей размерности, если она отсутствовала

Translation(cord, dr) вновь первый параметр **cord** - массив координат профиля, **dr** - вектор смещения профиля от начала координат в формате (dx, dy, dz):

Rotation(cord, angle, axis) **cord** - массив координат, **angle** - величина требуемого угла поворота в градусах, **axis** - ось, относительно которой нужно совершить поворот, в строковом формате 'x', 'y' или 'z'

Все функции можно последовательно применять, используя результат одной, как вход другой

In [4]:
def Scale(cord, chord):
    if len(cord[0,:]) < 3:
        cord = np.concat((cord, np.zeros((len(cord[:,0]),1))), axis=1) # если координаты на z нет, то добавляем
    cord_copy = np.copy(cord)
    n = len(cord[:,0])
    e = np.ones(n)
    cord_copy[:,0] *= chord*e
    cord_copy[:,1] *= chord*e
    return cord_copy

def Translation(cord, dr):
    dx, dy, dz = dr
    n = len(cord[:,0])
    e = np.ones(n)
    cord_copy = np.copy(cord)
    cord_copy[:,0] += dx*e
    cord_copy[:,1] += dy*e
    cord_copy[:,2] += dz*e
    return cord_copy
    
def Rotation(cord, angle, axis):
    phi = angle / 57.3
    if axis == 'x':
        S = np.array([[1, 0, 0],[0, np.cos(phi), np.sin(phi)], [0,-np.sin(phi) ,np.cos(phi)]])
    elif axis == 'y':
        S = np.array([[np.cos(phi), 0, -np.sin(phi)],[0, 1, 0], [np.sin(phi), 0, np.cos(phi)]])
    else:
        S = np.array([[np.cos(phi), np.sin(phi), 0], [-np.sin(phi), np.cos(phi), 0], [0,0,1]])
    
    cord_copy=np.copy(cord)
    for el in cord_copy:
        el @= S
    return cord_copy

Например отмасштабировали профиль до 220 мм длины хорды, потом сместили (никуду) и наконец сделали поворот вокруг оси OZ на -1 градус (то есть вращение против оси OZ). Потом сохраняем файл с запятой разделителем

In [ ]:
a = Scale(Coord, 220)
a = Translation(a, (0, 0, 0))
a = Rotation(a, -1, 'z')
np.savetxt('SD7032_220mm_1rot.txt', a, delimiter=',')